# HeartShare Table One — BDC runtime notebook (updated)

Builds a cross-study **Table 1** (plus cohort filters, subgroup comparisons,
and per-study longitudinal tables) and matching **boxplots / bar plots** from a
completed harmonization run.

This thin notebook reads and verifies the runtime archive recorded by the
harmonization run. By default, Step 1 automatically picks the **latest**
completed harmonization run under `/sbgenomics/workspace/output-files/` so you
can click **Run All** right after harmonizing (or pin an exact `RUN_DIR` for
reproducibility).

**How the numbers are computed**

| | |
|---|---|
| Continuous | median [Q1, Q3] by default (or mean ± SD via `CONTINUOUS_SUMMARIES`) |
| Categorical | n (%), where % is of participants **with an observed value** |
| Missing | measured against the study's participant total (`STUDY_TOTALS` or observed lower bound) |

> **Remember:** the harmonized long format emits *no row* for a missing
> measurement. Every variable carries its own `N` and `Missing` row. Set
> `STUDY_TOTALS` if you want exact enrolled N instead of observed lower bounds.


## 1. Select harmonization run (edit if needed)


In [ ]:
# ---------------------------------------------------------------------------
# 1. SELECT HARMONIZATION RUN — "latest" picks the newest completed run,
#    or set an exact folder path for reproducibility.
# ---------------------------------------------------------------------------
from pathlib import Path

HARMONIZATION_OUTPUT_ROOT = Path("/sbgenomics/workspace/output-files")

# "latest" automatically selects the newest completed harmonization run below.
# To pin a specific run, paste its full path, e.g.:
#   RUN_DIR = "/sbgenomics/workspace/output-files/harmonized_20260812T174618Z"
RUN_DIR = "latest"

# Optional full-path override for all Table One outputs.
# Leave as None to save inside <RUN_DIR>/<OUTPUT_SUBDIR> (configured in Step 3).
OUTPUT_DIR = None

def _is_completed_run(_path: Path) -> bool:
    if not _path.is_dir() or _path.name.endswith("_dryrun"):
        return False
    if not (_path / "manifest.json").is_file():
        return False
    return (
        (_path / "harmonized_long.parquet").is_file()
        or (_path / "harmonized_long.csv").is_file()
    )

_available_runs = sorted(
    (
        _path
        for _path in (HARMONIZATION_OUTPUT_ROOT.glob("harmonized_*") if HARMONIZATION_OUTPUT_ROOT.is_dir() else [])
        if _is_completed_run(_path)
    ),
    key=lambda _path: _path.stat().st_mtime,
    reverse=True,
)

print("Available completed harmonization runs (newest first):")
if _available_runs:
    for _idx, _path in enumerate(_available_runs[:20]):
        _marker = "  <-- latest" if _idx == 0 else ""
        print(f"  {_path}{_marker}")
else:
    print(f"  none found under {HARMONIZATION_OUTPUT_ROOT}")

if RUN_DIR == "latest":
    if not _available_runs:
        RUN_DIR = None
        print(
            "\nNo completed harmonization runs found. Run the harmonization notebook "
            "with DRY_RUN = False first, or set RUN_DIR to an exact folder path."
        )
    else:
        RUN_DIR = str(_available_runs[0])
        print(f"\nUsing latest run: {RUN_DIR}")
elif RUN_DIR:
    print(f"\nUsing pinned run: {RUN_DIR}")
else:
    print("\nSet RUN_DIR above to 'latest' or one of the folders listed, then rerun from Step 1.")

# === Advanced (usually leave as None) ======================================
# Optional verified runtime override for an older run or newer Table One features.
RUNTIME_BUNDLE_PATH = None
RUNTIME_BUNDLE_SHA256 = None


### Runtime setup (do not edit — run as-is)


In [ ]:
"""Small standard-library bootstrap used by HeartShare thin BDC notebooks."""

from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path
from zipfile import ZipFile


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def runtime_spec_from_release(manifest_path: Path) -> dict[str, object]:
    try:
        import yaml
    except ImportError as exc:
        raise RuntimeError("PyYAML is required to read the release manifest") from exc
    document = yaml.safe_load(Path(manifest_path).read_text(encoding="utf-8")) or {}
    retired = document.get("retired")
    if retired and os.environ.get("HEARTSHARE_ALLOW_RETIRED_RELEASE") != "1":
        raise RuntimeError(
            f"Release {document.get('release_version')} is RETIRED: {retired}. Use the release that "
            "supersedes it (set HEARTSHARE_ALLOW_RETIRED_RELEASE=1 only to reproduce an old result)."
        )
    spec = document.get("runtime_bundle")
    if not isinstance(spec, dict) or not spec.get("path") or not spec.get("sha256"):
        raise ValueError(
            f"Release manifest has no complete runtime_bundle path/checksum: {manifest_path}"
        )
    resolved = dict(spec)
    path = Path(str(spec["path"]))
    if not path.is_absolute():
        path = (Path(manifest_path).parent / path).resolve()
    resolved["path"] = path
    # What this release's mapping payload needs from its runtime (written by the release manager).
    resolved["required_runtime_features"] = list(document.get("required_runtime_features") or [])
    return resolved


def _safe_extract(archive_path: Path, target: Path) -> None:
    with ZipFile(archive_path) as archive:
        root = target.resolve()
        for member in archive.infolist():
            destination = (target / member.filename).resolve()
            if destination != root and root not in destination.parents:
                raise ValueError(f"Unsafe path in runtime archive: {member.filename}")
        archive.extractall(target)


def _verify_extracted(target: Path) -> dict[str, object]:
    manifest_path = target / "runtime_manifest.json"
    if not manifest_path.exists():
        raise ValueError("Runtime archive is missing runtime_manifest.json")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    for name, expected in (manifest.get("files") or {}).items():
        path = target / name
        if not path.is_file():
            raise ValueError(f"Runtime archive is missing declared file: {name}")
        actual = sha256_file(path)
        if actual != expected:
            raise ValueError(
                f"Runtime file checksum mismatch for {name}: expected {expected}, got {actual}"
            )
    return manifest


def _install_offline_wheels(target: Path, manifest: dict[str, object]) -> Path | None:
    requirements = list(manifest.get("required_offline_distributions") or [])
    if not requirements:
        return None
    wheelhouse = target / "wheelhouse"
    vendor = target / "vendor"
    marker = vendor / ".heartshare_wheels_installed"
    if marker.exists():
        return vendor
    vendor.mkdir(parents=True, exist_ok=True)
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "--no-index",
        "--find-links",
        str(wheelhouse),
        "--target",
        str(vendor),
        *requirements,
    ]
    subprocess.run(command, check=True)
    marker.write_text("ok\n", encoding="utf-8")
    return vendor


def bootstrap_runtime(
    archive_path: Path,
    expected_sha256: str,
    *,
    workspace_root: Path | None = None,
    expected_output_schema: str = "1.3",
    required_runtime_features: list[str] | tuple[str, ...] = (),
    supported_output_schemas: tuple[str, ...] | None = None,
) -> tuple[Path, dict[str, object]]:
    archive_path = Path(archive_path)
    if not archive_path.is_file():
        raise FileNotFoundError(f"Runtime bundle not found: {archive_path}")
    actual_sha256 = sha256_file(archive_path)
    if actual_sha256 != expected_sha256:
        raise ValueError(
            "Runtime bundle checksum mismatch: "
            f"expected {expected_sha256}, got {actual_sha256}"
        )

    workspace_root = Path(
        workspace_root
        or os.environ.get(
            "HEARTSHARE_VENDOR_ROOT", "/sbgenomics/workspace/_heartshare_vendor"
        )
    )
    workspace_root.mkdir(parents=True, exist_ok=True)
    target = workspace_root / actual_sha256[:16]
    if not (target / "runtime_manifest.json").exists():
        with tempfile.TemporaryDirectory(
            prefix="heartshare-runtime-", dir=workspace_root
        ) as temporary:
            staging = Path(temporary) / "unpacked"
            staging.mkdir()
            _safe_extract(archive_path, staging)
            _verify_extracted(staging)
            try:
                staging.rename(target)
            except FileExistsError:
                shutil.rmtree(staging)

    manifest = _verify_extracted(target)
    supported = supported_output_schemas or (expected_output_schema,)
    if str(manifest.get("output_schema_version")) not in supported:
        raise ValueError(
            "Runtime/output schema mismatch: "
            f"expected one of {supported}, runtime provides "
            f"{manifest.get('output_schema_version')}"
        )

    unsupported = sorted(set(required_runtime_features) - set(manifest.get("runtime_features") or ()))
    if unsupported:
        raise ValueError(
            "This release's mappings need runtime features the bundled runtime "
            f"({manifest.get('runtime_version')}) does not provide: {unsupported}. The release is "
            "mis-assembled — do not run it; rebuild it with a current runtime bundle."
        )

    vendor = _install_offline_wheels(target, manifest)
    import_root = target / str(manifest.get("import_root") or "bdc/src")
    for path in [vendor, target, import_root]:
        if path is not None and str(path) not in sys.path:
            sys.path.insert(0, str(path))

    os.environ["HEARTSHARE_RUNTIME_BUNDLE_PATH"] = str(archive_path)
    os.environ["HEARTSHARE_RUNTIME_BUNDLE_SHA256"] = actual_sha256
    os.environ["HEARTSHARE_RUNTIME_VERSION"] = str(
        manifest.get("runtime_version") or ""
    )
    os.environ["HEARTSHARE_RUNTIME_BUILD_GIT_SHA"] = str(
        manifest.get("build_git_sha") or ""
    )
    os.environ["HEARTSHARE_RUNTIME_BUILD_GIT_BRANCH"] = str(
        manifest.get("build_git_branch") or "runtime-bundle"
    )
    os.environ["HEARTSHARE_RUNTIME_BUILD_GIT_CLEAN"] = str(
        bool(manifest.get("build_git_clean", True))
    ).lower()
    return import_root, manifest


if not RUN_DIR:
    raise ValueError(
        "RUN_DIR is not set. In Step 1, choose one of the listed harmonization "
        "run folders, set RUN_DIR to its full path, and rerun from Step 1."
    )
_run_manifest_path = Path(RUN_DIR) / "manifest.json"
if not _run_manifest_path.is_file():
    raise FileNotFoundError(
        f"No harmonization manifest found at {_run_manifest_path}. RUN_DIR must "
        "be the exact output folder created by Step 5 of the harmonization "
        "notebook, not the output-files parent folder or the example path."
    )
_run_manifest = json.loads(_run_manifest_path.read_text(encoding="utf-8"))
_runtime_spec = dict(_run_manifest.get("runtime_bundle") or {})
_runtime_path = RUNTIME_BUNDLE_PATH or _runtime_spec.get("path")
_runtime_sha256 = RUNTIME_BUNDLE_SHA256 or _runtime_spec.get("sha256")
if not _runtime_path or not _runtime_sha256:
    raise ValueError(
        "This run does not record a runtime bundle. Set RUNTIME_BUNDLE_PATH and "
        "RUNTIME_BUNDLE_SHA256 in Step 1, or use heartshare_table_one.ipynb."
    )
_runtime_root, _runtime_manifest = bootstrap_runtime(
    Path(_runtime_path), str(_runtime_sha256),
    supported_output_schemas=("1.1", "1.2", "1.3"),
    required_runtime_features=("table_one/analysis/1",)
)
import table_one as t1
print(f"Runtime {_runtime_manifest['runtime_version']} ready")


### Load harmonization data (run as-is)


In [ ]:
# Loads the run data and dictionary. Nothing to edit here.
if not RUN_DIR:
    raise ValueError(
        "RUN_DIR is not set. In Step 1, set RUN_DIR = 'latest' or paste a "
        "completed harmonization run folder path, then rerun from Step 1."
    )

run_dir = Path(RUN_DIR)
if not run_dir.is_dir():
    raise FileNotFoundError(
        f"RUN_DIR does not exist: {run_dir}. Choose a folder listed in Step 1."
    )

files = t1.find_run_files(run_dir)
long_df = t1.load_long(files["long"])
variable_dictionary = t1.load_variable_dictionary(files["variable_dictionary"])

print(f"Run  : {run_dir}")
print(f"Data : {files['long'].name}  ({len(long_df):,} rows)")
print(f"Meta : {'variable_dictionary.csv' if variable_dictionary is not None else 'NOT FOUND — variable types will be inferred'}")


## 2. Inspect available studies, timepoints, and variables (run as-is)


In [ ]:
# ---------------------------------------------------------------------------
# 2. INSPECT AVAILABLE DATA — read-only. Lists studies, timepoints, variables,
#    and observed categorical values (useful for COHORT_FILTERS / STRATIFY).
# ---------------------------------------------------------------------------
_all_meta = t1.order_variables(
    t1.build_variable_metadata(long_df, variable_dictionary)
)

print(f"STUDIES ({long_df['study'].nunique()})")
print("-" * 72)
for _study, _block in long_df.groupby("study"):
    print(f'  "{_study}"{" " * max(1, 22 - len(str(_study)))} '
          f"{_block['participant_id'].nunique():>6,} participants   "
          f"{_block['standard_name'].nunique():>3} variables")

_timepoint_column = (
    "timepoint" if "timepoint" in long_df.columns and long_df["timepoint"].notna().any()
    else "exam_label" if "exam_label" in long_df.columns else None
)
if _timepoint_column:
    _timepoint_counts = (
        long_df.groupby(_timepoint_column, dropna=False)["participant_id"]
        .nunique()
        .sort_index()
    )
    print(f"\nTIMEPOINTS ({len(_timepoint_counts)})")
    print("-" * 72)
    for _timepoint, _count in _timepoint_counts.items():
        print(f'  "{_timepoint}"{" " * max(1, 28 - len(str(_timepoint)))} {_count:>6,} participants')
    if len(_timepoint_counts) > 1:
        print("\nPARTICIPANTS BY STUDY AND TIMEPOINT")
        print("-" * 72)
        _study_timepoints = (
            long_df.groupby(["study", _timepoint_column], dropna=False)
            ["participant_id"].nunique()
        )
        for (_study, _timepoint), _count in _study_timepoints.items():
            print(f"  {_study:<20} {str(_timepoint):<22} {_count:>6,}")

print(f"\nVARIABLES ({len(_all_meta)})")
print("-" * 72)
_current_domain = None
for _meta in _all_meta:
    if _meta.domain != _current_domain:
        _current_domain = _meta.domain
        print(f"\n  [{_current_domain or 'other'}]")
    _flag = "" if _meta.kind_source == "variable_dictionary" else " (type inferred)"
    _extra = ""
    if _meta.kind == "categorical":
        _vals = sorted(
            long_df.loc[long_df["standard_name"] == _meta.standard_name, "value"]
            .dropna()
            .astype(str)
            .unique()
        )
        if _vals:
            _preview_vals = ", ".join(_vals[:6])
            if len(_vals) > 6:
                _preview_vals += f", ... (+{len(_vals) - 6} more)"
            _extra = f"   values: [{_preview_vals}]"
    print(f'    "{_meta.standard_name}"{" " * max(1, 24 - len(_meta.standard_name))} '
          f"{_meta.kind:<12}{_flag}{_extra}")


## 3. Configure Table 1 (edit)


In [ ]:
# ---------------------------------------------------------------------------
# 3. CONFIGURE TABLE 1 — edit any settings below, then run Steps 3–6.
# ---------------------------------------------------------------------------

# Subfolder name inside RUN_DIR for this configuration's tables & figures.
# Change this (e.g. "table_one_bmi_ge30") when saving multiple analyses so
# earlier exports are not overwritten. Ignored if OUTPUT_DIR was set in Step 1.
OUTPUT_SUBDIR = "table_one"

# Single-visit table timepoint (or eligibility/subgroup visit in longitudinal mode).
# Set None only when the run contains exactly one timepoint.
TIMEPOINT = "baseline"

# Which variables, in the order you want them. Empty ([]) = every variable found.
VARIABLES = [
    # "age",
    # "sex",
    # "bmi",
]

# Which studies become columns. Empty ([]) = every study in the run.
STUDIES = []

# Optional participant filters evaluated at TIMEPOINT (combined with AND).
# Check exact categorical labels in Step 2 above.
COHORT_FILTERS = [
    # ("bmi", ">=", 30),
    # ("afib", "in", ["Yes", "1", "1.0"]),
]

# Optional side-by-side subgroup columns within each study.
STRATIFY = None
# STRATIFY = {"trait": "bmi", "cutoff": 30}
# STRATIFY = {"trait": "afib", "groups": {
#     "No AFib": ["No", "0", "0.0"], "AFib": ["Yes", "1", "1.0"]}}

# Separate tables and plots for each study across visits.
LONGITUDINAL = False
# Empty = all available visits per study, baseline first then duration/natural order.
TIMEPOINTS_BY_STUDY = {}
# TIMEPOINTS_BY_STUDY = {"STUDY-A": ["baseline", "week_08", "week_16"]}

# Default is median [IQR]; choose mean ± SD globally or per variable.
CONTINUOUS_SUMMARIES = {}
# CONTINUOUS_SUMMARIES = {"*": "median_iqr", "age": "mean_sd"}

# Known enrolled N per study (optional). Omitted studies use observed lower bounds.
STUDY_TOTALS = {
    # "HFN-NEAT": 100,
    # "TOPCAT": 3445,
}

# Add a pooled column across all studies.
INCLUDE_OVERALL = True

# Which files to write in Step 6. Remove any formats you don't need.
EXPORT_FORMATS = [
    "csv",
    "xlsx",
    "docx",
    "html",
]

# --- Quick validation before running Step 4 --------------------------------
_available_studies = sorted(long_df["study"].dropna().astype(str).unique())
_unknown_studies = sorted(set(STUDIES) - set(_available_studies))
if _unknown_studies:
    raise ValueError(f"Unknown STUDIES: {_unknown_studies}. Available: {_available_studies}")

_available_vars = sorted(long_df["standard_name"].dropna().astype(str).unique())
_unknown_vars = sorted(set(VARIABLES) - set(_available_vars))
if _unknown_vars:
    raise ValueError(f"Unknown VARIABLES: {_unknown_vars}. Available: {_available_vars}")

output_dir = Path(OUTPUT_DIR) if OUTPUT_DIR else run_dir / OUTPUT_SUBDIR
print(f"Output folder : {output_dir}")
if (output_dir / "analysis_settings.json").is_file():
    print(
        f"NOTE: {output_dir} already contains a previous Table One export. "
        "Change OUTPUT_SUBDIR above if you want to keep both."
    )


## 4. Build Table 1 (run as-is)


In [ ]:
# ---------------------------------------------------------------------------
# 4. BUILD TABLE 1 — computes participant selection and summary tables.
# ---------------------------------------------------------------------------
try:
    from IPython.display import display
except ImportError:
    display = print

if not hasattr(t1, "build_analysis"):
    raise RuntimeError(
        "This run's runtime predates subgroup/longitudinal Table One support. "
        "Use heartshare_table_one_updated.ipynb for this run, or set "
        "RUNTIME_BUNDLE_PATH and RUNTIME_BUNDLE_SHA256 in Step 1."
    )

output_dir = Path(OUTPUT_DIR) if OUTPUT_DIR else run_dir / OUTPUT_SUBDIR

try:
    analysis = t1.build_analysis(
        long_df, variable_dictionary, timepoint=TIMEPOINT, studies=STUDIES,
        variables=VARIABLES, cohort_filter=None, cohort_filters=COHORT_FILTERS,
        stratify=STRATIFY, longitudinal=LONGITUDINAL,
        timepoints_by_study=TIMEPOINTS_BY_STUDY, study_totals=STUDY_TOTALS,
        include_overall=INCLUDE_OVERALL, continuous_summaries=CONTINUOUS_SUMMARIES,
    )
except t1.DuplicateParticipantValuesError as _error:
    display(_error.report)
    raise

analysis_df = analysis.analysis_df
print("PARTICIPANT SELECTION (counts from harmonized data)")
display(analysis.selection)
for _panel in analysis.panels:
    print(_panel.title)
    display(_panel.result.table)
    for _warning in _panel.result.warnings:
        print(f"  - {_warning}")
    if not _panel.result.duplicate_report.empty:
        display(_panel.result.duplicate_report)

# Convenient handles for inspecting the first panel interactively.
result = analysis.panels[0].result
participant_values = analysis.panels[0].values


## 5. Figures (run as-is)


In [ ]:
# ---------------------------------------------------------------------------
# 5. FIGURES — saves all plots to disk; shows only overview plots inline by
#    default so large multi-variable runs do not freeze JupyterLab.
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt

MAKE_FIGURES = True
SHOW_INDIVIDUAL_PLOTS = False   # Set True to display every per-variable plot inline

saved = []
if not MAKE_FIGURES:
    print("MAKE_FIGURES = False — skipping figure generation.")
else:
    for _panel in analysis.panels:
        _panel_dir = output_dir if _panel.key == "table_one" else output_dir / _panel.key
        figure_dir = _panel_dir / "figures"
        figure_dir.mkdir(parents=True, exist_ok=True)
        for _meta in _panel.result.variables:
            if _meta.kind == "continuous":
                _figure = t1.plot_continuous(_panel.values, _meta, _panel.result.groups)
                _name = f"box_{t1.artifact_key(_meta)}.png"
            else:
                _figure = t1.plot_categorical(_panel.values, _meta, _panel.result.groups)
                _name = f"bar_{t1.artifact_key(_meta)}.png"
            _figure.axes[0].set_xlabel(_panel.axis_label)
            _figure.suptitle(_panel.title, fontsize=10)
            _figure.tight_layout()
            _figure.savefig(figure_dir / _name, bbox_inches="tight", facecolor="white")
            saved.append(str(figure_dir / _name))
            if SHOW_INDIVIDUAL_PLOTS:
                plt.show()
            plt.close(_figure)

            if LONGITUDINAL and _meta.kind == "continuous":
                _figure = t1.plot_visit_summary(_panel, _meta)
                _path = figure_dir / f"trend_{t1.artifact_key(_meta)}.png"
                _figure.savefig(_path, bbox_inches="tight", facecolor="white")
                saved.append(str(_path))
                if SHOW_INDIVIDUAL_PLOTS:
                    plt.show()
                plt.close(_figure)

        for _kind, _name in (("continuous", "overview_continuous.png"),
                             ("categorical", "overview_categorical.png")):
            _figure = t1.plot_overview(_panel.values, _panel.result.variables, _panel.result.groups, _kind)
            if _figure is None:
                continue
            _figure.suptitle(_panel.title, fontsize=10)
            _figure.tight_layout()
            _figure.savefig(figure_dir / _name, bbox_inches="tight", facecolor="white")
            saved.append(str(figure_dir / _name))
            plt.show()
            plt.close(_figure)

    print(f"Wrote {len(saved)} figure(s) under {output_dir}")
    if not SHOW_INDIVIDUAL_PLOTS and saved:
        print("Individual per-variable PNGs were saved to disk (set SHOW_INDIVIDUAL_PLOTS = True to view inline).")


## 6. Export tables and metadata (run as-is)


In [ ]:
# ---------------------------------------------------------------------------
# 6. EXPORT — writes requested table formats, settings, and selection counts.
# ---------------------------------------------------------------------------
written = {}
for _panel in analysis.panels:
    _panel_dir = output_dir if _panel.key == "table_one" else output_dir / _panel.key
    _paths = t1.export_table(
        _panel.result, _panel_dir, formats=EXPORT_FORMATS,
        title=f"Table 1. {_panel.title}",
    )
    _skipped = sorted(set(EXPORT_FORMATS) - set(_paths))
    for _format, _path in _paths.items():
        _key = _format if _panel.key == "table_one" else f"{_panel.key}/{_format}"
        written[_key] = _path
        print(f"  [{_format:<4}] {_path}")
    if _skipped:
        print(f"  Skipped format(s) for {_panel.key} (missing optional library): {_skipped}")

_sources = dict(files)
_sources["manifest"] = run_dir / "manifest.json"
t1.save_analysis_metadata(analysis, output_dir, _sources, EXPORT_FORMATS)
print(f"Settings         : {output_dir / 'analysis_settings.json'}")
print(f"Selection counts : {output_dir / 'participant_selection.csv'}")
